<div dir="rtl" style="text-align:right">
<h1>NB22F — آزمایش تجمیعی رتبه‌بندی محلی — v1</h1>
<p><strong>هدف ثابت: Recall@10 ≥ 85% و nDCG@10 ≥ 75%، با عدم افت Hit.</strong></p>
<p>این نوت‌بوک از کرنل مستقل اجرا می‌شود. NB22E یا نوت‌بوک‌های قبلی را اجرا نمی‌کند.
شش مسیر را با یک طرح مقایسه می‌کند: baseline، E5 مستقیم، ترکیب رتبهٔ واژه/سه‌حرفی،
ترکیب معنایی/واژه‌ای، مدل خطی رگرسیونی و مدل خطی زوجی.</p>
<p>تغییر اصلی: یادگیری ترکیب ویژگی‌های معنایی، واژه‌ای و رتبه‌ای از Development،
به‌جای تغییر مکرر عمق BGE. هیچ مدل دانلود یا اجرا نمی‌شود؛ بردارهای E5 قبلی استفاده می‌شوند.</p>
<p>تقسیم fit/validation/check در سطح مؤلفهٔ اسناد شاهد است. برچسب‌های check تا پس از
انتخاب یک روش در validation برای ارزیابی استفاده نمی‌شوند. چون Development قبلاً دیده شده،
این جداسازی پذیرش مستقل ریلیز نیست. احتمال کم‌بودن مؤلفه‌های کتاب صریحاً گزارش می‌شود.</p>
<p><strong>زمان اولیه کل: حدود ۵ تا ۳۰ دقیقه، وابسته به دستگاه؛ تضمین نیست.</strong>
ساخت ویژگی‌ها ممکن است بیشتر از پنج دقیقه باشد و در نوبت‌های ۱۲دقیقه‌ای ادامه‌پذیر است.
صرفاً سل F005 را در صورت PARTIAL دوباره اجرا کنید؛ Run All دوباره لازم نیست.</p>
<p>بدون API، دانلود، نصب پکیج، BGE، بازسازی embedding یا تغییر baseline.
فایل‌های مشتق‌شده در artifacts/phase2/quality_improvement/NB22F.v1 ذخیره می‌شوند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h3>F001 — تنظیمات ثابت و مکانیک امن؛ کمتر از پنج دقیقه</h3><p>نوت‌بوک را در notebooks/phase2 ذخیره و از ریشهٔ پروژه یا زیرپوشهٔ آن اجرا کنید. ورودی ریشه را فقط در صورت نیاز تغییر دهید.</p>
</div>

In [1]:
import os, json, math, hashlib, time, tempfile, collections, re, unicodedata, io, platform
from pathlib import Path
import numpy as np
from IPython.display import display, HTML

START = Path(os.environ.get("RAG_FINANCE_ROOT", str(Path.cwd()))).expanduser().resolve()
ROOT = next((p for p in (START, *START.parents) if (p/"notebooks/phase2").is_dir() and (p/"data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md").is_file()), None)
if ROOT is None: raise RuntimeError("PROJECT_ROOT_NOT_FOUND: set RAG_FINANCE_ROOT")
CFG = {"version":"NB22F.v1", "seed":"NB22F_COMPONENT_SPLIT_V1", "pool_depth":800,
       "session_seconds":720, "pause_per_query":0.01, "ridge_lambda":10.0,
       "pair_steps":400, "pair_batch":128, "pair_l2":0.01,
       "recall_target":0.85,"ndcg_target":0.75}
C056 = {"path":"reports/phase2/release/NB22.C056.retrieval_verification/C056.independent_retrieval_metric_verification.ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22.json", "sha256":"ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22"}
D_FINAL = {"path":"reports/phase2/quality_improvement/NB22D.v1/A05.final_report.4b1626f9669aa7bc8ffa0ada5ddffe8d7350aa8dcf5db9e956bd8c67f86c82de.json","sha256":"4b1626f9669aa7bc8ffa0ada5ddffe8d7350aa8dcf5db9e956bd8c67f86c82de"}
OUT = ROOT/"artifacts/phase2/quality_improvement/NB22F.v1"
OUT.mkdir(parents=True,exist_ok=True)
def require(ok,reason):
    if not ok: raise RuntimeError(reason)
def canonical(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"),allow_nan=False).encode()
def sha(raw): return hashlib.sha256(raw).hexdigest()
def locate(relative):
    p=Path(relative);require(not p.is_absolute() and ".." not in p.parts,"RELATIVE_PATH_REQUIRED")
    out=(ROOT/p).resolve();require(out.is_relative_to(ROOT),"OUTSIDE_PROJECT");return out
def bh(b):
    h={b[k] for k in ("sha256","file_bytes_sha256") if k in b};require(len(h)==1,"AMBIGUOUS_HASH");return next(iter(h))
def checked(b):
    p=locate(b["path"]);a=p.stat();raw=p.read_bytes();z=p.stat()
    require((a.st_size,a.st_mtime_ns,a.st_ino)==(z.st_size,z.st_mtime_ns,z.st_ino),"INPUT_CHANGED")
    require(sha(raw)==bh(b),"HASH_MISMATCH:"+b["path"]);return raw
def read(b): return json.loads(checked(b))
def rows(b): return [json.loads(x) for x in checked(b).splitlines() if x.strip()]
def keyed(rr,key):
    d={}
    for r in rr: require(r[key] not in d,"DUPLICATE_ID");d[r[key]]=r
    return d
def atomic(p,raw):
    fd,tmp=tempfile.mkstemp(dir=p.parent,prefix=".pending-")
    try:
        with os.fdopen(fd,"wb") as f:f.write(raw);f.flush();os.fsync(f.fileno())
        os.replace(tmp,p)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)
    require(p.read_bytes()==raw,"WRITE_RELOAD_CONFLICT")
def save(stem,value):
    raw=canonical(value);p=OUT/(stem+"."+sha(raw)+".json");atomic(p,raw)
    return {"path":p.relative_to(ROOT).as_posix(),"sha256":sha(raw),"bytes":len(raw),"reload_validation":"PASS"}
def metric(order,gold):
    require(bool(gold) and len(order)==len(set(order)),"INVALID_METRIC_INPUT")
    flags=[int(c in gold) for c in order[:10]]
    ideal=math.fsum(1/math.log2(i+2) for i in range(min(10,len(gold))))
    return {"recall_at_10":sum(flags)/len(gold),"ndcg_at_10":math.fsum(v/math.log2(i+2) for i,v in enumerate(flags))/ideal,"hit_at_10":float(any(flags))}
def mean(rr):
    rr=list(rr);require(bool(rr),"EMPTY_COHORT")
    return {k:math.fsum(r[k] for r in rr)/len(rr) for k in rr[0]}
def words(text):
    text=unicodedata.normalize("NFKC",text).lower().translate(str.maketrans("يك٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹","یک01234567890123456789"))
    return re.findall(r"[^\W_]+", "".join(c for c in text if not unicodedata.combining(c) and c!="ـ").replace("\u200c"," "))
print(json.dumps({"root":str(ROOT),"version":CFG["version"],"api_calls":0,"downloads":False,"model_inference":False},ensure_ascii=False))


{"root": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance", "version": "NB22F.v1", "api_calls": 0, "downloads": false, "model_inference": false}


<div dir="rtl" style="text-align:right">
<h3>F002 — ورودی‌های پین‌شده و بازخوانی baseline؛ کمتر از پنج دقیقه</h3><p>همهٔ ۲۶۶۴ سؤال Development استفاده می‌شوند؛ Test فقط برای شناسایی split خوانده می‌شود و سنجیده نمی‌شود.</p>
</div>

In [2]:
verified=read(C056);dfinal=read(D_FINAL);bindings=verified["input_file_receipts"]
CHUNKS=keyed(rows(bindings["chunks"]),"chunk_id")
GOLDEN=keyed(rows(bindings["golden"]),"candidate_id")
HYBRID=keyed(rows(bindings["hybrid"]),"candidate_id")
IDS=sorted(q for q in GOLDEN if HYBRID[q]["split"]=="development")
require(len(IDS)==2664 and len(CHUNKS)==3488,"POPULATION_CONFLICT")
BASE_ROWS=keyed(rows(dfinal["retained_rankings"]),"candidate_id")
POOL_ROWS=keyed(rows(dfinal["pool_artifact"]),"candidate_id")
channel_bindings={(r["path"],bh(r)):r for r in dfinal["artifacts"] if r["path"].split("/")[-1].startswith("A03.lexical_channels.")}
require(len(channel_bindings)==1,"LEXICAL_BINDING_REQUIRED")
CHANNEL_BINDING=next(iter(channel_bindings.values()))
CHANNELS=keyed(rows(CHANNEL_BINDING),"candidate_id")
require(set(BASE_ROWS)==set(POOL_ROWS)==set(CHANNELS)==set(IDS),"DEV_COVERAGE_CONFLICT")
locator=collections.defaultdict(set)
for cid,r in CHUNKS.items():
    for m in r["member_locators"]:locator[canonical(m["locator"])].add(cid)
GOLD={};BASE={};POOLS={}
for q in IDS:
    gg=set()
    for e in GOLDEN[q]["evidence_locators"]:
        matches=locator[canonical(e["source_locator"])]
        require(matches and any(CHUNKS[c]["source_type"] in GOLDEN[q]["source_types"] for c in matches),"GOLD_LOCATOR_CONFLICT")
        gg.update(matches)
    GOLD[q]=gg
    r=BASE_ROWS[q];order=r["ranked_chunk_ids"]
    require(r["split"]==POOL_ROWS[q]["split"]=="development","SPLIT_CONFLICT")
    require(len(order)==len(set(order)) and set(order).issubset(CHUNKS),"BASE_ORDER_CONFLICT")
    mm=metric(order,gg);require(all(abs(mm[k]-r["metrics"][k])<1e-9 for k in mm),"BASE_METRIC_REPLAY_CONFLICT")
    pool=POOL_ROWS[q]["candidate_chunk_ids"]
    require(len(pool)==len(set(pool)) and len(pool)<=800 and set(pool).issubset(CHUNKS),"POOL_CONFLICT")
    BASE[q]=order;POOLS[q]=list(dict.fromkeys(order[:40]+pool))
print(json.dumps({"baseline_replay":"PASS","dev_queries":len(IDS),"pool_rule":"baseline_top40_union_saved_pool","max_candidates":max(map(len,POOLS.values()))},ensure_ascii=False))


{"baseline_replay": "PASS", "dev_queries": 2664, "pool_rule": "baseline_top40_union_saved_pool", "max_candidates": 800}


<div dir="rtl" style="text-align:right">
<h3>F003 — تقسیم مؤلفه‌ای و ثبت طرح؛ کمتر از پنج دقیقه</h3><p>برای هر نوع منبع، مؤلفه‌ها با هش ثابت به fit/validation/check تخصیص می‌یابند. مؤلفهٔ مشترک بین بخش‌ها وجود ندارد. این داده‌ها از قبل دیده‌شده‌اند و holdout مستقل نام نمی‌گیرند. دادهٔ fit کم یا نبود شاهد مثبت در pool باعث توقف می‌شود.</p>
</div>

In [3]:
parent={q:q for q in IDS}
def find(q):
    while parent[q]!=q:parent[q]=parent[parent[q]];q=parent[q]
    return q
seen={}
for q in IDS:
    for c in GOLD[q]:
        sid=CHUNKS[c]["source_id"]
        if sid in seen:
            a,b=find(q),find(seen[sid]);parent[max(a,b)]=min(a,b)
        else:seen[sid]=q
COMP={q:find(q) for q in IDS};groups=collections.defaultdict(list)
for q in IDS:groups[COMP[q]].append(q)
strata=collections.defaultdict(list)
for c,qs in groups.items():
    label=','.join(sorted({s for q in qs for s in GOLDEN[q]["source_types"]}));strata[label].append(c)
PARTS={"fit":[],"validation":[],"check":[]};component_counts={}
for label,cs in sorted(strata.items()):
    cs=sorted(cs,key=lambda c:(sha((CFG["seed"]+label+c).encode()),c));n=len(cs)
    require(n>=3,"INSUFFICIENT_COMPONENTS_FOR_THREE_WAY_SPLIT:"+label)
    nv=max(1,int(n*.2));nc=max(1,int(n*.2));nf=n-nv-nc
    require(nf>=1,"NO_FIT_COMPONENT")
    assigned={"fit":cs[:nf],"validation":cs[nf:nf+nv],"check":cs[nf+nv:]}
    component_counts[label]={k:len(v) for k,v in assigned.items()}
    for name,subset in assigned.items():PARTS[name].extend(q for c in subset for q in groups[c])
for name in PARTS:PARTS[name].sort()
require(set().union(*map(set,PARTS.values()))==set(IDS),"PARTITION_COVERAGE")
require(sum(map(len,PARTS.values()))==len(IDS),"PARTITION_OVERLAP")
PLAN={"configuration":CFG,"input_bindings":bindings,"baseline":dfinal["retained_rankings"],"pool":dfinal["pool_artifact"],"channels":CHANNEL_BINDING,"partitions":PARTS,"component_counts":component_counts,
"routes":["baseline","dense_only","word_char_rrf","dense_word_char_rrf","linear_ridge","pairwise_linear"],
"selection":"validation: strict nDCG gain and nondecreasing Recall/Hit vs baseline; else baseline; one winner evaluated on check",
"scope":"PREVIOUSLY_EXPOSED_DEVELOPMENT_NOT_INDEPENDENT_RELEASE_ACCEPTANCE","release_acceptance_granted":False}
PLAN_RECEIPT=save("F003.plan",PLAN)
print(json.dumps({"queries":{k:len(v) for k,v in PARTS.items()},"component_counts":component_counts,"plan":PLAN_RECEIPT},ensure_ascii=False,indent=2))


{
  "queries": {
    "fit": 1507,
    "validation": 533,
    "check": 624
  },
  "component_counts": {
    "book": {
      "fit": 1,
      "validation": 1,
      "check": 1
    },
    "transcript": {
      "fit": 97,
      "validation": 32,
      "check": 32
    },
    "website": {
      "fit": 262,
      "validation": 86,
      "check": 86
    }
  },
  "plan": {
    "path": "artifacts/phase2/quality_improvement/NB22F.v1/F003.plan.0e5b4205ee6d489ea52b8db01771a75c710bb6caca8c937110d0a5b8d4d2fd93.json",
    "sha256": "0e5b4205ee6d489ea52b8db01771a75c710bb6caca8c937110d0a5b8d4d2fd93",
    "bytes": 127864,
    "reload_validation": "PASS"
  }
}


<div dir="rtl" style="text-align:right">
<h3>F004 — مصرف بردارهای موجود؛ معمولاً کمتر از پنج دقیقه</h3><p>بدون Qdrant، اجرای مدل یا ساخت embedding. هش بردار سؤال‌ها کنترل می‌شود. اتصال passage با همان قاعدهٔ NB14 است. هش فایل‌های corpus ثبت می‌شود؛ تطابق inventory قدیمی جداگانه ادعا نمی‌شود.</p>
</div>

In [4]:
handoff=read({"path":"artifacts/phase2/embedding/nb14_dense_index_handoff_v1.json","sha256":"77eb9f34e36f174cf748fd99caa1d507b5dc8e4abcb51f62b452c35d46688efc"})
qmeta=read({"path":"artifacts/phase2/hybrid_retrieval/query_embedding_cache_v1/development_query_embeddings.json","sha256":"bc97f189d5cadb801f5aaaad9a86715509ad2a91a18dc4840b0918c0c8f2568e"})
require(qmeta["candidate_ids"]==IDS,"QUERY_ORDER_CONFLICT")
require(qmeta["contract"]["model_revision"]==handoff["embedding_config"]["revision"]=="614241f622f53c4eeff9890bdc4f31cfecc418b3","MODEL_REVISION_CONFLICT")
require(qmeta["contract"]["model_id"]==handoff["embedding_config"]["model_id"]=="intfloat/multilingual-e5-small" and qmeta["contract"]["normalize_embeddings"] is True and qmeta["contract"]["query_prefix"]=="query:","EMBEDDING_CONTRACT_CONFLICT")
h=hashlib.sha256()
for q in IDS:
    raw=(q+'\x1f'+GOLDEN[q]["evaluation_content"]["question"]).encode();h.update(len(raw).to_bytes(8,'big'));h.update(raw)
require(h.hexdigest()==qmeta["contract"]["query_set_sha256"],"QUESTION_TEXT_CONFLICT")
QV=np.load(io.BytesIO(checked({"path":qmeta["embedding_path"],"sha256":qmeta["embedding_file_sha256"]})),allow_pickle=False)
require(QV.shape==(2664,384) and QV.dtype==np.float32 and np.isfinite(QV).all(),"QUERY_VECTOR_SHAPE")
require(np.max(np.abs(np.linalg.norm(QV,axis=1)-1))<1e-4,"QUERY_NORM")
CID=sorted(CHUNKS);CPOS={c:i for i,c in enumerate(CID)};QPOS={q:i for i,q in enumerate(IDS)}
pkeys={c:'passage:'+sha(' '.join(str(CHUNKS[c]['content']).split()).encode()) for c in CID};needed=set(pkeys.values())
vc={};seenkeys=set();receipts=[];cache_root=locate(handoff['vector_cache']['root']);release=read(handoff['embedding_release']);files=sorted(cache_root.rglob('*.npz'))
require(len(files)==release['cache']['npz_file_count'],"CACHE_FILE_COUNT")
for p in files:
    raw=p.read_bytes();receipts.append({'path':p.relative_to(ROOT).as_posix(),'sha256':sha(raw)})
    with np.load(io.BytesIO(raw),allow_pickle=False) as a:
        kk=a['input_keys'];vv=a['embeddings'];require(kk.ndim==1 and vv.shape==(len(kk),384) and vv.dtype==np.float32 and np.isfinite(vv).all(),"CACHE_ARRAY_CONFLICT")
        for k,v in zip(kk,vv):
            k=str(k);require(k not in seenkeys,"DUPLICATE_CACHE_KEY");seenkeys.add(k)
            if k in needed:vc[k]=v.copy()
    time.sleep(.01)
require(needed.issubset(vc),"PASSAGE_VECTOR_MISSING_NO_RECOMPUTATION")
CV=np.stack([vc[pkeys[c]] for c in CID]);del vc
require(np.max(np.abs(np.linalg.norm(CV,axis=1)-1))<1e-4,"PASSAGE_NORM")
VECTOR_RECEIPT=save('F004.vector_receipts',{'files':receipts,'query_file':qmeta['embedding_file_sha256'],'original_corpus_inventory_verified':False})
FEATURE_ID=sha(canonical({'plan':PLAN_RECEIPT,'vectors':VECTOR_RECEIPT,'features_version':'V1'}))
FEATURE_DIR=OUT/('features_'+FEATURE_ID);FEATURE_DIR.mkdir(exist_ok=True)
TOK={c:collections.Counter(words(CHUNKS[c]['content'])) for c in CID}
SETS={c:set(t) for c,t in TOK.items()};NUMS={c:{x for x in t if any(ch.isdigit() for ch in x)} for c,t in SETS.items()}
print('PASS: existing vectors bound to questions and passages; no model execution')


PASS: existing vectors bound to questions and passages; no model execution


<div dir="rtl" style="text-align:right">
<h3>F005 — ساخت ویژگی‌های همهٔ مسیرها؛ ممکن است بیش از پنج دقیقه باشد</h3><p>تخمین اولیه ۵ تا ۲۰ دقیقه. هر نوبت حداکثر حدود ۱۲ دقیقه؛ پس از PARTIAL فقط همین سل را تکرار کنید. ویژگی‌ها: شباهت E5، رتبهٔ baseline/word/char/hybrid، پوشش واژه و عدد، فراوانی واژه و طول قطعه. برچسب‌ها در فایل ویژگی نوشته نمی‌شوند.</p>
</div>

In [5]:
FEATURE_NAMES=['dense_cosine','baseline_rr','word_rr','char_rr','hybrid_rr','word_coverage','numeric_coverage','term_frequency','log_length','baseline_present','hybrid_present']
def feature_path(q):return FEATURE_DIR/(sha(q.encode())+'.json')
def load_feature(q):
    r=json.loads(feature_path(q).read_bytes());require(r['feature_id']==FEATURE_ID and r['query_id']==q and r['checksum']==sha(canonical(r['payload'])),"FEATURE_CACHE_CONFLICT")
    require(r['payload']['ids']==POOLS[q],"FEATURE_CANDIDATE_CONFLICT")
    x=np.asarray(r['payload']['x'],dtype=np.float64);require(x.shape==(len(POOLS[q]),len(FEATURE_NAMES)) and np.isfinite(x).all(),"FEATURE_SHAPE");return x
start=time.monotonic();fresh=0;reused=0;FEATURES_READY=False
for i,q in enumerate(IDS):
    if time.monotonic()-start>=CFG['session_seconds']:break
    if feature_path(q).is_file():load_feature(q);reused+=1
    else:
        ranks={name:{c:j+1 for j,c in enumerate(order)} for name,order in [('baseline',BASE[q]),('word',CHANNELS[q]['channels']['word']),('char',CHANNELS[q]['channels']['char']),('hybrid',[r['chunk_id'] for r in HYBRID[q]['candidates']])]}
        qt=set(words(GOLDEN[q]['evaluation_content']['question']));qn={t for t in qt if any(z.isdigit() for z in t)}
        dense=np.einsum('ij,j->i',CV,QV[QPOS[q]],optimize=False);xx=[]
        for c in POOLS[q]:
            rr=[1/(60+ranks[name][c]) if c in ranks[name] else 0 for name in ('baseline','word','char','hybrid')]
            xx.append([float(dense[CPOS[c]]),*rr,len(qt&SETS[c])/max(1,len(qt)),len(qn&NUMS[c])/max(1,len(qn)),sum(min(TOK[c].get(t,0),3) for t in qt)/max(1,len(qt)),math.log1p(sum(TOK[c].values())),float(c in ranks['baseline']),float(c in ranks['hybrid'])])
        payload={'ids':POOLS[q],'x':xx};atomic(feature_path(q),canonical({'feature_id':FEATURE_ID,'query_id':q,'payload':payload,'checksum':sha(canonical(payload))}));fresh+=1
        time.sleep(CFG['pause_per_query'])
    if (i+1)%50==0:print(json.dumps({'completed':i+1,'total':len(IDS),'fresh':fresh,'reused':reused,'seconds':round(time.monotonic()-start,1)}),flush=True)
FEATURES_READY=all(feature_path(q).is_file() for q in IDS)
print(json.dumps({'status':'COMPLETE' if FEATURES_READY else 'PARTIAL_RESUMABLE_REPEAT_F005_ONLY','fresh':fresh,'reused':reused,'seconds':round(time.monotonic()-start,1)}))


{"completed": 50, "total": 2664, "fresh": 50, "reused": 0, "seconds": 2.4}
{"completed": 100, "total": 2664, "fresh": 100, "reused": 0, "seconds": 4.4}
{"completed": 150, "total": 2664, "fresh": 150, "reused": 0, "seconds": 6.4}
{"completed": 200, "total": 2664, "fresh": 200, "reused": 0, "seconds": 8.2}
{"completed": 250, "total": 2664, "fresh": 250, "reused": 0, "seconds": 10.0}
{"completed": 300, "total": 2664, "fresh": 300, "reused": 0, "seconds": 11.9}
{"completed": 350, "total": 2664, "fresh": 350, "reused": 0, "seconds": 13.9}
{"completed": 400, "total": 2664, "fresh": 400, "reused": 0, "seconds": 15.8}
{"completed": 450, "total": 2664, "fresh": 450, "reused": 0, "seconds": 17.6}
{"completed": 500, "total": 2664, "fresh": 500, "reused": 0, "seconds": 19.6}
{"completed": 550, "total": 2664, "fresh": 550, "reused": 0, "seconds": 21.4}
{"completed": 600, "total": 2664, "fresh": 600, "reused": 0, "seconds": 23.2}
{"completed": 650, "total": 2664, "fresh": 650, "reused": 0, "seconds"

<div dir="rtl" style="text-align:right">
<h3>F006 — آموزش دو مدل کوچک محلی؛ تخمین کمتر از پنج دقیقه</h3><p>مدل خطی ridge و یادگیری زوجی فقط روی fit آموزش می‌بینند؛ استانداردسازی هم فقط fit است. پارامترها ثابت‌اند و جست‌وجو ندارند. برای جلوگیری از فشار BLAS، محاسبات کوچکِ ماتریسی با یک رشته انجام می‌شود؛ بستهٔ جدید نصب نمی‌شود.</p>
</div>

In [6]:
require(FEATURES_READY,"RUN_F005_UNTIL_COMPLETE")
# Optional thread control must be available before training; never installs packages.
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    raise RuntimeError('threadpoolctl unavailable: no training started; do not install automatically')
rng=np.random.default_rng(22801);train_x=[];train_y=[]
for q in PARTS['fit']:
    ids=POOLS[q];pos=[i for i,c in enumerate(ids) if c in GOLD[q]]
    if not pos:continue
    neg=[i for i,c in enumerate(ids) if c not in GOLD[q]]
    chosen_neg=list(dict.fromkeys(neg[:20]+(rng.choice(neg,size=min(20,len(neg)),replace=False).tolist() if neg else [])))
    chosen=pos+chosen_neg;x=load_feature(q)
    train_x.append(x[chosen]);train_y.extend([1.0]*len(pos)+[0.0]*len(chosen_neg))
require(bool(train_x),"NO_FIT_POSITIVES")
X=np.vstack(train_x);y=np.asarray(train_y);mu=X.mean(axis=0);sd=X.std(axis=0);sd[sd<1e-8]=1
Z=(X-mu)/sd
with threadpool_limits(limits=1):
    aug=np.column_stack([np.ones(len(Z)),Z]);reg=CFG['ridge_lambda']*np.eye(aug.shape[1]);reg[0,0]=0
    ridge=np.linalg.solve(aug.T@aug+reg,aug.T@y)
# Pairwise training: fixed one positive/negative draw per step item.
pair_groups=[]
for q in PARTS['fit']:
    ids=POOLS[q];p=[i for i,c in enumerate(ids) if c in GOLD[q]];n=[i for i,c in enumerate(ids) if c not in GOLD[q]]
    if p and n:
        chosen_n=list(dict.fromkeys(n[:20]+rng.choice(n,size=min(20,len(n)),replace=False).tolist()))
        x=(load_feature(q)[p+chosen_n]-mu)/sd
        pair_groups.append((x,len(p)))
require(bool(pair_groups),"NO_PAIRWISE_TRAINING_GROUPS")
w=np.zeros(len(FEATURE_NAMES))
with threadpool_limits(limits=1):
    for step in range(CFG['pair_steps']):
        batch=[]
        for _ in range(CFG['pair_batch']):
            x,np_=pair_groups[int(rng.integers(len(pair_groups)))];pi=int(rng.integers(np_));ni=int(rng.integers(np_,len(x)))
            batch.append(x[pi]-x[ni])
        delta=np.asarray(batch);margin=np.clip(delta@w,-30,30)
        grad=-(delta/(1+np.exp(margin))[:,None]).mean(axis=0)+CFG['pair_l2']*w
        w-=.05/math.sqrt(1+step/100)*grad
        if step%50==0:print(json.dumps({'pair_step':step,'total':CFG['pair_steps']}),flush=True);time.sleep(.02)
MODEL_RECEIPT=save('F006.linear_models',{'feature_id':FEATURE_ID,'features':FEATURE_NAMES,'mu':mu.tolist(),'sd':sd.tolist(),'ridge':ridge.tolist(),'pairwise':w.tolist(),'fit_query_count':len(PARTS['fit']),'fit_rows':len(y),'fit_queries_with_positive_and_negative':len(pair_groups)})
print(json.dumps({'training':'COMPLETE','model':MODEL_RECEIPT,'neural_model_inference':False}))


{"pair_step": 0, "total": 400}
{"pair_step": 50, "total": 400}
{"pair_step": 100, "total": 400}
{"pair_step": 150, "total": 400}
{"pair_step": 200, "total": 400}
{"pair_step": 250, "total": 400}
{"pair_step": 300, "total": 400}
{"pair_step": 350, "total": 400}
{"training": "COMPLETE", "model": {"path": "artifacts/phase2/quality_improvement/NB22F.v1/F006.linear_models.8e82a1961e11a69d10306032fcc70ffdf8b30b1bc2b546a988edc8ea6357dcf1.json", "sha256": "8e82a1961e11a69d10306032fcc70ffdf8b30b1bc2b546a988edc8ea6357dcf1", "bytes": 1287, "reload_validation": "PASS"}, "neural_model_inference": false}


<div dir="rtl" style="text-align:right">
<h3>F007 — مقایسهٔ validation و انتخاب یک مسیر؛ ممکن است بیش از پنج دقیقه باشد</h3><p>تخمین ۱ تا ۱۰ دقیقه. شش مسیر فقط روی validation مقایسه می‌شوند؛ baseline در هر صورت موجود است. شرط انتخاب: nDCG بیشتر و Recall/Hit غیرکاهشی. عبور از حدود مطلق جدا گزارش می‌شود؛ انتخاب خودکار به معنی تغییر محصول نیست.</p>
</div>

In [7]:
ROUTES=['baseline','dense_only','word_char_rrf','dense_word_char_rrf','linear_ridge','pairwise_linear']
def route_orders(q):
    ids=POOLS[q];x=load_feature(q);z=(x-mu)/sd
    with threadpool_limits(limits=1):
        ridge_scores=ridge[0]+z@ridge[1:];pair_scores=z@w
    positions={c:i for i,c in enumerate(ids)}
    dense_ranks={c:i+1 for i,c in enumerate(sorted(ids,key=lambda c:(-x[positions[c],0],c)))}
    scores={'dense_only':x[:,0], 'word_char_rrf':x[:,2]+x[:,3],
            'dense_word_char_rrf':np.array([1/(60+dense_ranks[c]) for c in ids])+x[:,2]+x[:,3],
            'linear_ridge':ridge_scores,'pairwise_linear':pair_scores}
    return {'baseline':BASE[q],**{name:sorted(ids,key=lambda c:(-float(v[i_map[c]]),c)) for name,v in scores.items() for i_map in [{c:i for i,c in enumerate(ids)}]}}
def table(results,before,title):
    tr=[]
    for name,r in results.items():
        for k,label,t in [('recall_at_10','Recall@10',.85),('ndcg_at_10','nDCG@10',.75),('hit_at_10','Hit@10',None)]:
            mark='✅' if t is not None and r[k]>=t else '';target='تعیین نشده' if t is None else f'{t:.0%}'
            delta=100*(r[k]-before[k]);delta=0 if abs(delta)<1e-10 else delta
            tr.append(f'<tr><td>{name}</td><td>{label}</td><td>{before[k]:.2%}</td><td>{r[k]:.2%}</td><td>{delta:+.2f}</td><td>{target}</td><td>{mark}</td></tr>')
    display(HTML('<div dir="rtl" style="text-align:right"><p>'+title+'</p><table><tr><th>روش</th><th>معیار</th><th>قبل</th><th>بعد</th><th>تغییر؛ واحد درصد</th><th>حد</th><th>قبول عددی</th></tr>'+''.join(tr)+'</table></div>'))
VAL_VALUES={r:{} for r in ROUTES}
for i,q in enumerate(PARTS['validation']):
    oo=route_orders(q)
    for r in ROUTES:VAL_VALUES[r][q]=metric(oo[r],GOLD[q])
    if (i+1)%50==0:print(json.dumps({'validation_completed':i+1,'total':len(PARTS['validation'])}),flush=True)
    time.sleep(.005)
VAL={r:mean(v.values()) for r,v in VAL_VALUES.items()};vbase=VAL['baseline']
qualified=[r for r in ROUTES if r!='baseline' and VAL[r]['ndcg_at_10']>vbase['ndcg_at_10']+1e-12 and all(VAL[r][k]>=vbase[k]-1e-12 for k in ['recall_at_10','hit_at_10'])]
WINNER=sorted(qualified,key=lambda r:(-VAL[r]['ndcg_at_10'],r))[0] if qualified else 'baseline'
FREEZE=save('F007.validation_selection',{'winner':WINNER,'validation_metrics':VAL,'qualified':qualified,'plan':PLAN_RECEIPT,'models':MODEL_RECEIPT,'feature_id':FEATURE_ID,'check_used_for_selection':False,'release_acceptance_granted':False})
table(VAL,vbase,'مقایسهٔ validation؛ دادهٔ Development دیده‌شده، نه پذیرش مستقل')
print(json.dumps({'frozen_winner':WINNER,'selection':FREEZE},ensure_ascii=False,indent=2))


{"validation_completed": 50, "total": 533}
{"validation_completed": 100, "total": 533}
{"validation_completed": 150, "total": 533}
{"validation_completed": 200, "total": 533}
{"validation_completed": 250, "total": 533}
{"validation_completed": 300, "total": 533}
{"validation_completed": 350, "total": 533}
{"validation_completed": 400, "total": 533}
{"validation_completed": 450, "total": 533}
{"validation_completed": 500, "total": 533}


روش,معیار,قبل,بعد,تغییر؛ واحد درصد,حد,قبول عددی
baseline,Recall@10,66.52%,66.52%,+0.00,85%,
baseline,nDCG@10,59.99%,59.99%,+0.00,75%,
baseline,Hit@10,86.12%,86.12%,+0.00,تعیین نشده,
dense_only,Recall@10,66.52%,44.73%,-21.79,85%,
dense_only,nDCG@10,59.99%,35.60%,-24.39,75%,
dense_only,Hit@10,86.12%,58.91%,-27.20,تعیین نشده,
word_char_rrf,Recall@10,66.52%,55.70%,-10.82,85%,
word_char_rrf,nDCG@10,59.99%,48.30%,-11.69,75%,
word_char_rrf,Hit@10,86.12%,72.80%,-13.32,تعیین نشده,
dense_word_char_rrf,Recall@10,66.52%,58.46%,-8.06,85%,


{
  "frozen_winner": "baseline",
  "selection": {
    "path": "artifacts/phase2/quality_improvement/NB22F.v1/F007.validation_selection.184cf95466c02210a43e74a5c24d27a3210bbd58337e03377e5a8b6b695427ab.json",
    "sha256": "184cf95466c02210a43e74a5c24d27a3210bbd58337e03377e5a8b6b695427ab",
    "bytes": 1430,
    "reload_validation": "PASS"
  }
}


<div dir="rtl" style="text-align:right">
<h3>F008 — بررسی یک گزینهٔ ثابت و جمع‌بندی؛ ممکن است بیش از پنج دقیقه باشد</h3><p>تخمین ۱ تا ۱۰ دقیقه. فقط روش منتخب validation و baseline روی check سنجیده می‌شوند. اگر نتیجه ضعیف شد، گزینهٔ دوم از check انتخاب نمی‌شود. خروجی شامل افت سؤال‌ها، برش منابع، سقف pool و شرط عددی است؛ تغییر محصول انجام نمی‌شود.</p>
</div>

In [8]:
frozen=read(FREEZE);require(frozen['winner']==WINNER,'SELECTION_CHANGED')
CHECK_BEFORE={};CHECK_AFTER={};CHECK_ORDERS={}
for i,q in enumerate(PARTS['check']):
    if WINNER=='baseline':order=BASE[q]
    else:order=route_orders(q)[WINNER]
    CHECK_ORDERS[q]=order;CHECK_BEFORE[q]=metric(BASE[q],GOLD[q]);CHECK_AFTER[q]=metric(order,GOLD[q])
    if (i+1)%50==0:print(json.dumps({'check_completed':i+1,'total':len(PARTS['check'])}),flush=True)
    time.sleep(.005)
before=mean(CHECK_BEFORE.values());after=mean(CHECK_AFTER.values())
no_loss=all(after[k]>=before[k]-1e-12 for k in before)
point_pass=WINNER!='baseline' and no_loss and after['ndcg_at_10']>before['ndcg_at_10']+1e-12 and after['recall_at_10']>=.85 and after['ndcg_at_10']>=.75
slices={}
for source in ('book','transcript','website'):
    qs=[q for q in PARTS['check'] if source in GOLDEN[q]['source_types']]
    slices[source]={'queries':len(qs),'before':mean(CHECK_BEFORE[q] for q in qs) if qs else None,'after':mean(CHECK_AFTER[q] for q in qs) if qs else None,'source_components':len({COMP[q] for q in qs})}
changes={k:{'improved':sum(CHECK_AFTER[q][k]>CHECK_BEFORE[q][k]+1e-12 for q in CHECK_AFTER),'worsened':sum(CHECK_AFTER[q][k]<CHECK_BEFORE[q][k]-1e-12 for q in CHECK_AFTER)} for k in before}
FINAL={'version':'NB22F.v1','status':'EXPERIMENT_COMPLETE','selected_method':WINNER,'before':before,'after':after,'source_slices':slices,'query_changes':changes,'check_queries':len(PARTS['check']),'check_components':len({COMP[q] for q in PARTS['check']}),'point_conditions_met':point_pass,'no_regression':no_loss,
'pool_recall_ceiling':math.fsum(min(10,len(set(POOLS[q])&GOLD[q]))/len(GOLD[q]) for q in PARTS['check'])/len(PARTS['check']),
'scope':'INTERNAL_COMPONENT_SEPARATED_PREVIOUSLY_EXPOSED_DEVELOPMENT','independent_release_acceptance':False,'release_acceptance_granted':False,'baseline_replaced':False,'new_api_calls':0,'downloads':False,'new_embeddings':0,'neural_model_pairs':0,'plan':PLAN_RECEIPT,'models':MODEL_RECEIPT,'selection':FREEZE,'vector_receipts':VECTOR_RECEIPT,
'limits':['No independent holdout claim','Small number of book components can dominate uncertainty','Corpus vector hashes recorded; old inventory hash not independently reproduced','No generation/citation/safety release qualification in this notebook'],
'next_step':'REVIEW_FIXED_CHECK_RESULT; NO_RUNNER_UP_SELECTION_OR_AUTOMATIC_NEW_EXPERIMENT'}
FINAL['orders_artifact']=save('F008.check_orders',{'orders':CHECK_ORDERS,'before_metrics':CHECK_BEFORE,'after_metrics':CHECK_AFTER})
FINAL_RECEIPT=save('F008.final_report',FINAL)
table({WINNER:after},before,'نتیجهٔ گزینهٔ ثابت روی check داخلی؛ تیک عددی پذیرش ریلیز نیست')
print(json.dumps({**FINAL,'report':FINAL_RECEIPT},ensure_ascii=False,indent=2))


{"check_completed": 50, "total": 624}
{"check_completed": 100, "total": 624}
{"check_completed": 150, "total": 624}
{"check_completed": 200, "total": 624}
{"check_completed": 250, "total": 624}
{"check_completed": 300, "total": 624}
{"check_completed": 350, "total": 624}
{"check_completed": 400, "total": 624}
{"check_completed": 450, "total": 624}
{"check_completed": 500, "total": 624}
{"check_completed": 550, "total": 624}
{"check_completed": 600, "total": 624}


روش,معیار,قبل,بعد,تغییر؛ واحد درصد,حد,قبول عددی
baseline,Recall@10,69.60%,69.60%,+0.00,85%,
baseline,nDCG@10,60.38%,60.38%,+0.00,75%,
baseline,Hit@10,84.78%,84.78%,+0.00,تعیین نشده,


{
  "version": "NB22F.v1",
  "status": "EXPERIMENT_COMPLETE",
  "selected_method": "baseline",
  "before": {
    "recall_at_10": 0.6959706959706959,
    "ndcg_at_10": 0.6038440986404439,
    "hit_at_10": 0.8477564102564102
  },
  "after": {
    "recall_at_10": 0.6959706959706959,
    "ndcg_at_10": 0.6038440986404439,
    "hit_at_10": 0.8477564102564102
  },
  "source_slices": {
    "book": {
      "queries": 148,
      "before": {
        "recall_at_10": 0.8277027027027027,
        "ndcg_at_10": 0.6828021570976294,
        "hit_at_10": 0.8513513513513513
      },
      "after": {
        "recall_at_10": 0.8277027027027027,
        "ndcg_at_10": 0.6828021570976294,
        "hit_at_10": 0.8513513513513513
      },
      "source_components": 1
    },
    "transcript": {
      "queries": 309,
      "before": {
        "recall_at_10": 0.6181229773462783,
        "ndcg_at_10": 0.5152317046957774,
        "hit_at_10": 0.7864077669902912
      },
      "after": {
        "recall_at_10": 0.6181

<div dir="rtl" style="text-align:right">
<h3>راهنمای توقف، ادامه و Git</h3><p>F005 پس از PARTIAL فقط خودش تکرار می‌شود. قطع در F006 تا F008 به کش ویژگی‌ها آسیب نمی‌زند؛ همان مرحله را دوباره اجرا کنید. هیچ فایل مبنا حذف نمی‌شود. فایل‌های مشتق‌شده در پوشهٔ artifacts هستند و طبق gitignore فعلی محلی می‌مانند. برای نگهداری گزارش نهایی، رسید چاپ‌شدهٔ F008 و خود گزارش را جداگانه در بستهٔ شواهد نگه دارید؛ این نوت‌بوک Git را تغییر نمی‌دهد.</p><p>این نسخه در محیط سازنده از نظر ساختار و منطق با دادهٔ مصنوعی بررسی شده است؛ داده‌ها و کش واقعی مک در این محیط موجود نیستند. زمان و کیفیت واقعی تنها پس از اجرای محلی معلوم می‌شوند.</p>
</div>